# Winning Jeopardy!

"Jeopardy!" is a popular TV quiz show in the US where participants answer questions to win money. It has been on the air since 1964 and has grown to be a major force in US Popular Culture.

## Aim

The goal of this project is to analyze a dataset of past Jeopardy! questions to gain insights into any patterns that may exist in an effort to aid contestants in preparing for an appearance on the show. 

The dataset was acquired from the Subreddit [r/datasets](https://www.reddit.com/r/datasets/comments/1uyd0t/200000_jeopardy_questions_in_a_json_file/), comprising over 216,930 Jeopardy questions from shows aired between 1984 and 2012. The original data was scraped from [J-Archive](https://www.j-archive.com/index.php) a fan-made archive of Jeopardy! games.

## Exploring and Cleaning the Dataset

In [1]:
import pandas as pd
import numpy as np

pd.options.display.max_colwidth = None

jeopardy = pd.read_csv('JEOPARDY_CSV.csv')

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams


In [2]:
jeopardy.columns

Index(['Show Number', ' Air Date', ' Round', ' Category', ' Value',
       ' Question', ' Answer'],
      dtype='str')

Some column names containg trailing whitespace. This can be removed to ensure consistent column names.

In [3]:
jeopardy.columns = [col.strip() for col in jeopardy.columns]
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='str')

In [4]:
jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Show Number  216930 non-null  int64
 1   Air Date     216930 non-null  str  
 2   Round        216930 non-null  str  
 3   Category     216930 non-null  str  
 4   Value        213296 non-null  str  
 5   Question     216930 non-null  str  
 6   Answer       216927 non-null  str  
dtypes: int64(1), str(6)
memory usage: 11.6 MB


There seem to be three missing answers, this warrants further investigation.

In [5]:
jeopardy[jeopardy["Answer"].isnull()]

,Show Number,Air Date,Round,Category,Value,Question,Answer
94817,4346,2003-06-23,Jeopardy!,"GOING ""N""SANE",$200,"It often precedes ""and void""",NaN
143297,6177,2011-06-21,Double Jeopardy!,NOTHING,$400,"This word for ""nothing"" precedes ""and void"" to mean ""not valid""",NaN
178922,4573,2004-06-23,Jeopardy!,MUCH ADO ABOUT NOTHING,$200,"Completes the title of the 1939 book by Agatha Christie ""And Then There Were...""",NaN


Funnily, the missing answers are "Null" and "None" - pandas has set these to `np.nan` values by default. This can be rectified by simply reassigning the correct answers as strings.

In [6]:
jeopardy.iloc[94817,6] = "Null"
jeopardy.iloc[143297,6] = "Null"
jeopardy.iloc[178922,6] = "None"

jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Show Number  216930 non-null  int64
 1   Air Date     216930 non-null  str  
 2   Round        216930 non-null  str  
 3   Category     216930 non-null  str  
 4   Value        213296 non-null  str  
 5   Question     216930 non-null  str  
 6   Answer       216930 non-null  str  
dtypes: int64(1), str(6)
memory usage: 11.6 MB


We also observe a number of missing values in the `Value` field cells. Looking deeper, all of these rows correspond to questions asked in the **final round** of the show. This seems to be the standard format for the show - final round questions are not worth any prize money.

In [7]:
jeopardy[jeopardy["Value"].isnull()].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer
137559,3235,1998-10-02,Final Jeopardy!,SPORTS AUTHORS,NaN,"Hemingway described this writer's 1961 book ""Out of My League"" as ""The dark side of... Walter Mitty""",George Plimpton
173546,5736,2009-07-13,Final Jeopardy!,AMERICAN HISTORY,NaN,The area that's now the State of Indiana was acquired in this war,the Revolutionary War
214664,5560,2008-11-07,Final Jeopardy!,PRESIDENTIAL ELECTIONS,NaN,One of the 2 presidents to win the national popular vote 3 times but only be elected president twice,(1 of) Grover Cleveland & Andrew Jackson
84631,4469,2004-01-29,Final Jeopardy!,MYTHOLOGY,NaN,"They were the 2 parents of a son who ended up half man, half woman",Hermes & Aphrodite
38787,4721,2005-02-28,Final Jeopardy!,COLLEGE LIBRARIES,NaN,"Built in memory of a victim of this tragedy, Harvard's Widener Library was opened in 1915",the sinking of the Titanic
69638,5074,2006-10-05,Final Jeopardy!,CHILDREN'S LIT,NaN,"This Roald Dahl book begins, ""These two very old people are the father and mother of Mr. Bucket""",Charlie and the Chocolate Factory
124055,4202,2002-12-03,Final Jeopardy!,"WASHINGTON, D.C.",NaN,The National Mall is bounded by these 2 avenues whose names recall historic documents,Constitution & Independence Avenues
53013,3550,2000-01-28,Final Jeopardy!,FRONT PAGE HISTORY,NaN,"An August 6, 1945 Associated Press story described this as a ""Japanese army base""",Hiroshima (story about the dropping of the first atomic bomb)
89059,5875,2010-03-12,Final Jeopardy!,FILM LEGENDS,NaN,His only competitive Oscar win was for Best Score in 1973 for a 1952 film in which he had starred as a washed-up comic,Charlie Chaplin
18738,4844,2005-10-06,Final Jeopardy!,PUBLICATIONS,NaN,"In 1889 a daily New York news summary called the ""Customers' Afternoon Letter"" became this publication",The Wall Street Journal


### Standardizing Question and Answer Fields

It is essential that punctuation is removed and the case of each letter in the `Question` and `Answer` fields is standardized such that all instances of a word are treated equally (for example, "The" and "the" should be recognized as the same word).

The `Value` column must also be cleaned by converting the value to an integer and setting null values to 0.

Cleaning functions can be applied to each column to achieve this.

In [8]:
import re

def clean_text(text):
    
    text = str(text).lower()                # Convert to string and lowercase
    text = re.sub(r"[^\w\s]", "", text)     # Replace non-alphanumeric characters and whitespace with nothing - this removes punctuation
    text = re.sub(r"\s+", " ", text)        # Replace multiple spaces with single space
    return text

def clean_value(value):
    
    value = str(value)
    value = re.sub(r"[$]", "", value) # Remove dollar sign
    
    try:
        value = int(value)            # Convert to integer
        
    except Exception:
        value = 0                     # Handles non-numeric values like NaN, None, setting them to 0
        
    return value
  
jeopardy["Clean Question"] = jeopardy["Question"].apply(clean_text)

jeopardy["Clean Answer"] = jeopardy["Answer"].apply(clean_text)

jeopardy["Clean Value"] = jeopardy["Value"].apply(clean_value)

jeopardy["Air Date"] = pd.to_datetime(jeopardy["Air Date"])  # Convert to datetime

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus,for the last 8 years of his life galileo was under house arrest for espousing this mans theory,copernicus,200
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe,no 2 1912 olympian football star at carlisle indian school 6 mlb seasons with the reds giants braves,jim thorpe,200
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona,the city of yuma in this state has a record average of 4055 hours of sunshine each year,arizona,200
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's,in 1963 live on the art linkletter show this company served its billionth burger,mcdonalds,200
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams,signer of the dec of indep framer of the constitution of mass second president of the united states,john adams,200


In [9]:
jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Show Number     216930 non-null  int64         
 1   Air Date        216930 non-null  datetime64[us]
 2   Round           216930 non-null  str           
 3   Category        216930 non-null  str           
 4   Value           213296 non-null  str           
 5   Question        216930 non-null  str           
 6   Answer          216930 non-null  str           
 7   Clean Question  216930 non-null  str           
 8   Clean Answer    216930 non-null  str           
 9   Clean Value     216930 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(7)
memory usage: 16.6 MB


## How Often do Answers Appear in Questions?

In [10]:
def answer_words_in_question(row):
  
  """Returns the proportion of words in the answer that are also present in the question."""
  
  question = row["Clean Question"]
  answer = row["Clean Answer"]
  
  split_question = question.split(" ")
  split_answer = answer.split(" ")
  
  match_count = 0
  
  if "the" in split_answer:          # Remove "the" and "a" from answer words as these are common to both questions and answers, but do not have use in answering our analysis question.
    split_answer.remove("the")
    
  if "a" in split_answer:
    split_answer.remove("a")
    
  if len(split_answer) == 0:
    return 0
  
  for word in split_answer:
    if word in split_question:
      match_count += 1
    
  return match_count / len(split_answer) 

jeopardy["Answer in Question"] = jeopardy.apply(answer_words_in_question, axis=1)

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus,for the last 8 years of his life galileo was under house arrest for espousing this mans theory,copernicus,200,0.0
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe,no 2 1912 olympian football star at carlisle indian school 6 mlb seasons with the reds giants braves,jim thorpe,200,0.0
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona,the city of yuma in this state has a record average of 4055 hours of sunshine each year,arizona,200,0.0
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's,in 1963 live on the art linkletter show this company served its billionth burger,mcdonalds,200,0.0
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams,signer of the dec of indep framer of the constitution of mass second president of the united states,john adams,200,0.0


In [11]:
answer_in_question_counts = jeopardy["Answer in Question"].value_counts()

answer_in_question_counts

Answer in Question
0.000000    196666
0.500000     10780
0.333333      4051
0.250000      1461
1.000000      1414
0.666667       760
0.200000       608
0.400000       280
0.166667       251
0.142857       116
0.750000       108
0.285714        71
0.600000        70
0.125000        57
0.428571        29
0.222222        27
0.375000        26
0.800000        25
0.111111        21
0.571429        19
0.300000         8
0.714286         7
0.833333         7
0.100000         7
0.181818         6
0.272727         5
0.083333         5
0.625000         5
0.857143         4
0.153846         4
0.230769         4
0.777778         2
0.444444         2
0.583333         2
0.555556         2
0.545455         2
0.363636         2
0.090909         2
0.350000         1
0.636364         1
0.700000         1
0.307692         1
0.368421         1
0.071429         1
0.133333         1
0.117647         1
0.818182         1
0.454545         1
0.066667         1
0.266667         1
0.058824         1
0.384615    

The majority of answers do not contain any words featured in the question. 

In [12]:
number_of_questions_containing_words_in_answer = np.sum([count for index, count in enumerate(answer_in_question_counts) if index > 0])

print(f"""The number of questions which contain words in the answer: {number_of_questions_containing_words_in_answer:,}
      
This makes up {number_of_questions_containing_words_in_answer / len(jeopardy) * 100:.2f}% of the total questions.""")

The number of questions which contain words in the answer: 20,264

This makes up 9.34% of the total questions.


The fact that only around 9% of the answers in the dataset have words featured in the question means that this avenue of analysis should certainly not be the focus for preparing a contestant to win the show. This does not however mean that there is no merit in looking into this further.

Below, insights are uncovered for questions that do have answers that are partly or fully featured in the question.

In [13]:
jeopardy[jeopardy["Answer in Question"] > 0].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
44208,5010,2006-05-26,Double Jeopardy!,LATIN AMERICAN LAKES & RIVERS,$2000,This longest Venezuelan river starts in the Parima Highlands near the Brazilian border,the Orinoco River,this longest venezuelan river starts in the parima highlands near the brazilian border,the orinoco river,2000,0.500000
26916,4892,2005-12-13,Jeopardy!,ROUGH ALTERNATE LITERARY ENDINGS,$400,"Jenny told me ""Love means never having to say you're sorry"". But now she was gone, & that nurse was giving me the eye.",Love Story,jenny told me love means never having to say youre sorry but now she was gone that nurse was giving me the eye,love story,400,0.500000
132690,6215,2011-09-30,Jeopardy!,BLU-RAYS,$600,"Time travel info is on the Blu-ray of this Jake Gyllenhaal film, & you don't have to access the title ""Code"" to get it",Source Code,time travel info is on the bluray of this jake gyllenhaal film you dont have to access the title code to get it,source code,600,0.500000
67615,4543,2004-05-12,Jeopardy!,THE CONSTITUTION,$200,"If no candidate receives a majority of electoral votes, then this body shall elect the president",the House of Representatives,if no candidate receives a majority of electoral votes then this body shall elect the president,the house of representatives,200,0.333333
85095,3659,2000-06-29,Double Jeopardy!,THEATRE,$600,"Of ""Closer"", ""Side Man"", ""Lonesome West"" or ""Not About Nightingales"", 1999's Tony winner for Best New Play","""Side Man""",of closer side man lonesome west or not about nightingales 1999s tony winner for best new play,side man,600,1.000000
15221,5408,2008-02-27,Jeopardy!,A BILLION REASONS,$400,"According to its website, this chain serves more than a billion ""finger lickin' good"" chicken dinners annually",Kentucky Fried Chicken,according to its website this chain serves more than a billion finger lickin good chicken dinners annually,kentucky fried chicken,400,0.333333
156975,3253,1998-10-28,Jeopardy!,NEWSPAPER NAMES,$500,This Tulsa morning paper is a translation of Germany's Die Welt,The Tulsa World,this tulsa morning paper is a translation of germanys die welt,the tulsa world,500,0.500000
21191,369,1986-02-06,Jeopardy!,ODDS & ENDS,$200,The winner of the Miss USA contest goes on to represent the U.S. in this pageant,Miss Universe,the winner of the miss usa contest goes on to represent the us in this pageant,miss universe,200,0.500000
141408,6006,2010-10-25,Double Jeopardy!,MUMMIES OF THE WORLD,$1600,"Found in the family vault still wearing his boots, mummified German Baron von Holz died during this early 17th c. war",the Thirty Years' War,found in the family vault still wearing his boots mummified german baron von holz died during this early 17th c war,the thirty years war,1600,0.333333
111129,6284,2012-01-05,Jeopardy!,EMMY-WINNING TV,$200,"""Hill Street Station"" (Writing, 1981)",Hill Street Blues,hill street station writing 1981,hill street blues,200,0.666667


Looking at a sample of answers containing some proportion of their words in the question, it does not seem to be particularly helpful in devising a strategy. In many cases, it is already given in the question that the answer will contain some words from the question. 

For instance: "French film star Simone Signoret married 2 men named Yves: director Yves Allegret & this actor" - we can simply deduce the answer will be two words long "Yves ___" without any prior knowledge.

It is an interesting observation however to look into the `HIDDEN COUNTRIES` category. This appears to be a style of question that features a country name hidden within the question statement. An example from the dataset: "There are many who talk in dialect in this country" - the answer is the hidden country name **India**. It would therefore be wise to become familiar with country names and practice this style of question.

In [14]:
jeopardy[jeopardy["Category"] == "HIDDEN COUNTRIES"].sample(5)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
82682,4914,2006-01-12,Jeopardy!,HIDDEN COUNTRIES,$800,I've been through analysis,Ghana,ive been through analysis,ghana,800,0.00
130281,5883,2010-03-24,Jeopardy!,HIDDEN COUNTRIES,$1000,Give a dog a bone & he'll bark gratefully,Gabon (in dog a bone),give a dog a bone hell bark gratefully,gabon in dog a bone,1000,0.50
82688,4914,2006-01-12,Jeopardy!,HIDDEN COUNTRIES,$1000,Hug and accept it,Uganda,hug and accept it,uganda,1000,0.00
82670,4914,2006-01-12,Jeopardy!,HIDDEN COUNTRIES,$400,Chair and table,Iran,chair and table,iran,400,0.00
173871,4760,2005-04-22,Jeopardy!,HIDDEN COUNTRIES,$1000,Give a dog a bone in this country & you've made a friend for life,Gabon (in DOG A BONE),give a dog a bone in this country youve made a friend for life,gabon in dog a bone,1000,0.75


Above, we also found 1,414 of the answers in the dataset are fully contained within the question. It could be worth looking into this to aid in preparation for winning the show.

In [15]:
jeopardy[jeopardy["Answer in Question"] == 1].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
48572,3883,2001-06-20,Double Jeopardy!,WHICH CAME FIRST?,$400,"""All My Children"", ""All in the Family"", ""All-Star Anything Goes""",All My Children,all my children all in the family allstar anything goes,all my children,400,1.0
97170,4859,2005-10-27,Jeopardy!,THE SMALLEST IN AREA,$1000,"South Africa, Zimbabwe, Namibia",Zimbabwe,south africa zimbabwe namibia,zimbabwe,1000,1.0
178312,4904,2005-12-29,Double Jeopardy!,NOT A BRITISH PRIME MINISTER,$1200,"Balmoral, Bentinck, Baldwin",Balmoral,balmoral bentinck baldwin,balmoral,1200,1.0
162658,4193,2002-11-20,Jeopardy!,BIOLOGY TEST,$400,The only hormone that can lower blood sugar is: (A) epinephrine (B) glucagon (C) thyroxine (D) insulin,insulin,the only hormone that can lower blood sugar is a epinephrine b glucagon c thyroxine d insulin,insulin,400,1.0
96753,3870,2001-06-01,Jeopardy!,TV OR NOT TV,$500,"""Make 'Em Pay"", ""Make Me Laugh"", ""Make The Grade""",Make 'Em Pay,make em pay make me laugh make the grade,make em pay,500,1.0
149852,2048,1993-06-30,Double Jeopardy!,LANGUAGES,$400,"Of Balinese, Bengali or Bulgarian, the one that has the most speakers by far",Bengali,of balinese bengali or bulgarian the one that has the most speakers by far,bengali,400,1.0
173809,4939,2006-02-16,Jeopardy!,THE LARGEST U.S. STATE,$1000,"Michigan, Minnesota, Mississippi",Michigan,michigan minnesota mississippi,michigan,1000,1.0
29141,5605,2009-01-09,Jeopardy!,THE MOST POPULOUS NATION,$800,"Venezuela, Uruguay, Colombia",Colombia,venezuela uruguay colombia,colombia,800,1.0
81586,4697,2005-01-25,Jeopardy!,THE SMALLEST IN AREA,$800,"Sweden, Norway, Denmark",Denmark,sweden norway denmark,denmark,800,1.0
30581,5612,2009-01-20,Jeopardy!,STUPID GEOGRAPHIC ANSWERS,$1000,Kansas City is at the confluence of the Missouri River & this river,the Kansas River,kansas city is at the confluence of the missouri river this river,the kansas river,1000,1.0


Some categories such as `THE LARGEST IN AREA` list three countries and the contestant must choose the correct answer. It certainly seems worthwhile to study up on geographical general knowledge for this category (and others such as `NOT A NATIONAL CAPITAL`).

Moving past this, the `STUPID ANSWERS` category looks interesting:

In [16]:
avg = jeopardy[jeopardy["Category"] == "STUPID ANSWERS"]["Answer in Question"].mean()

print(f"""The average proportion of answer words in the question for the category "STUPID ANSWERS": {avg:.2f}""")

jeopardy[jeopardy["Category"] == "STUPID ANSWERS"].sample(5)

The average proportion of answer words in the question for the category "STUPID ANSWERS": 0.66


,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
20068,3009,1997-10-02,Jeopardy!,STUPID ANSWERS,$400,"After Mexico, it's the most populous country whose official language is Spanish",Spain,after mexico its the most populous country whose official language is spanish,spain,400,0.0
20056,3009,1997-10-02,Jeopardy!,STUPID ANSWERS,$200,Network on which you'd see David Duchovny play Fox Mulder & Matthew Fox play Charlie Salinger,FOX,network on which youd see david duchovny play fox mulder matthew fox play charlie salinger,fox,200,1.0
119222,4459,2004-01-15,Jeopardy!,STUPID ANSWERS,$600,"In 2003 top model Tyra Banks conducted a search for ""America's Next"" one of these",Top Model,in 2003 top model tyra banks conducted a search for americas next one of these,top model,600,1.0
187109,4651,2004-11-22,Jeopardy!,STUPID ANSWERS,$200,In 2002 Melissa Gilbert was reelected president of this guild formed by actors of the silver screen,the Screen Actors Guild,in 2002 melissa gilbert was reelected president of this guild formed by actors of the silver screen,the screen actors guild,200,1.0
82964,3639,2000-06-01,Jeopardy!,STUPID ANSWERS,$100,In 1567 Pope Pius V issued a papal bull against the fighting of these,Bulls,in 1567 pope pius v issued a papal bull against the fighting of these,bulls,100,0.0


This category seems to reward quick thinking/instinct over thoughtfulness. Often, the answer to the question is present in the question itself. This seems to be a style of question for contestants to become familiar with and practice.

## Recycled Questions

A second avenue of interest is looking into how often questions are reused across the show. This cannot be fully answered using this dataset of questions, as it does not fully encompass every single question that has ever been asked on the show over its 62 year runtime. Nevertheless, insights can still be garnered from analysis of this sample.

Below, a `Question Overlap` column is defined. 

>What this is attempting to describe is, for a given question, what is the proportion of single word terms (words that are six or more characters long) that have appeared in previous questions asked on the show?

The reasoning behind restricting the words in the question to be six or more characters long is to filter out filler words such as "the" and "than". These are common, but do not provide information about any particular question.

In [17]:
question_overlap = []

terms_previously_used = set()                       # Words that are 6 or more letters long will be stored in the set.

jeopardy = jeopardy.sort_values("Air Date")

for i, row in jeopardy.iterrows():
  split_question = row["Clean Question"].split(" ")
  split_question = [word for word in split_question if len(word) > 5]  # Remove words that are less than 6 letters long
  match_count = 0       
  
  for word in split_question:
    if word in terms_previously_used:
      match_count += 1                              # If word in question is also in terms_previously_used, increment match_count.
    
  for word in split_question:                       # Add words in question to terms_previously_used. This must be placed after the if statement above to avoid over-counting.
    terms_previously_used.add(word) 
     
  if len(split_question) > 0:
    match_count /= len(split_question)
      
  question_overlap.append(match_count)
      
jeopardy["Question Overlap"] = question_overlap

In [18]:
jeopardy["Question Overlap"].mean()

np.float64(0.872176637774269)

This tells us for the average Jeopardy! question, ~87% of words that are six-letters or longer have been used in previous questions. This finding is not very significant since it only considers single word terms. Still, it may mean it is worth looking into this further.

---

Below, we attempt to approximate the proportion of questions that have appeared on the show previously. We assume for a question in a given category, if the answer to this question is identical to the answer of another question that has appeared in the same category, the questions are either the same or very similar.

The code below:

* Iterates through every row in the dataset. For each question, a `category_answer_pair` tuple is created which stores the question's category and answer.
* If the category and answer for the current question in the loop is already contained within the set of previous category-answer pairs, the question will be deemed similar to a previously asked question and the corresponding question, category and answer is appended to the list of `similar_questions`.
* The `similar_questions` list of tuples is then converted to a dataframe and unpacked into category, question and answer columns for further inspection.

In [19]:
similar_questions = []
previous_category_answer_pairs = set()

for i, row in jeopardy.iterrows():
  
  question = row["Question"]
  category = row["Category"]
  answer = row["Clean Answer"]
  category_answer_pair = (category, answer)
  
  if (row["Category"], row["Clean Answer"]) in previous_category_answer_pairs:
    similar_questions.append((category, question, answer))
    
  previous_category_answer_pairs.add(category_answer_pair)

similar_questions = pd.Series(similar_questions).to_frame(name="Category-Question Pair")

similar_questions["Category"] = similar_questions["Category-Question Pair"].apply(lambda x: x[0])
similar_questions["Question"] = similar_questions["Category-Question Pair"].apply(lambda x: x[1])
similar_questions["Answer"] = similar_questions["Category-Question Pair"].apply(lambda x: x[2])

similar_questions.drop("Category-Question Pair", axis=1, inplace=True)

grouped = similar_questions.groupby(["Category", "Question"]).agg({"Answer": "first"})

In [20]:
proportion_of_similar_questions = len(similar_questions) / len(jeopardy)

print(f"The percentage of questions in the sample that are either the same as or similar to another question: {proportion_of_similar_questions * 100:.2f}%")

The percentage of questions in the sample that are either the same as or similar to another question: 4.91%


It has been found that just under 5% of the questions contained in the dataset are the same as or highly similar to question(s) asked previously. This is a fairly significant proportion - we can approximate **just under 1 in 20 questions have been asked previously in some form or another** (assuming the format of the show and style of questions have not changed significantly since this dataset was acquired).

---

In [21]:
grouped.head(30)

Answer
Category           Question                                                                                                                                                                                
"A" IN GEOGRAPHY   It's the capital of Jordan                                                                                                                                                         amman
                   Known to the Romans as Numidia, this large African country borders the Mediterranean Sea                                                                                         algeria
                   This Scottish seaport lies between the rivers Dee & Don                                                                                                                         aberdeen
                   This ancient city is the capital of Greece                                                                                                                                        athens
                   This country controls the eastern half of Tierra del Fuego, largest island in an archipelago of the same name                                                                  argentina
"A" MEN            1994's "Three Tall Women" earned him his third Pulitzer Prize for Drama                                                                                                     edward albee
                   This famed fashion photographer passed away in October 2004                                                                                                               richard avedon
"A" PLUS           Whether his name is Bud or not, he's the head man at a monastery                                                                                                                   abbot
"A.C."             His Third Symphony includes his earlier "Fanfare for the Common Man"                                                                                                       aaron copland
                   The U.S. Navy has 12 of these equipped with steam-driven catapults                                                                                                     aircraft carriers
"AA"               From the Arabic, it's a low bow, or a salutation meaning "peace"                                                                                                                  salaam
"AW", SHUCKS       Developed by sailors to pass the time, <a href="http://www.j-archive.com/media/2011-06-14_J_21.jpg" target="_blank">it</a>'s the art of carving on whalebone or ivory          scrimshaw
"B" IN FASHION     For many women their wardrobe includes a black one of these jackets as well as a double breasted navy one                                                                       a blazer
"B" IN GEOGRAPHY   The NFL Europe's Dragons play their home games in this Spanish city that hosted the 1992 Summer Olympics                                                                       barcelona
"B" MOVIES         A dimwitted shut-in becomes the toast of Washington, D.C. society in this comedy starring Peter Sellers                                                                      being there
"B" SHARP          A joke says that when you play country music this way, your wife, your dog & your car return                                                                                   backwards
                   From the Italian for "jest", it's a clown or a fool                                                                                                                            a buffoon
                   Gaborone is the capital of this southern African country                                                                                                                        botswana
                   Sepia & mahogany are tones of this color                                                          

The above questions have been deemed to be similar or identical to previous questions asked on the show. We can investigate further to verify if this is indeed the case. 

Taking for example the question with the answer of `bellerophon` in the above dataframe - we can check the full dataset for instances where bellerophon is also the correct answer. We expect to see very similar questions.

In [22]:
jeopardy[jeopardy["Clean Answer"] == "bellerophon"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
170603,3831,2001-04-09,Double Jeopardy!,"""B"" SHARP",$600,This hero rode Pegasus,Bellerophon,this hero rode pegasus,bellerophon,600,0.0,1.0
173010,5338,2007-11-21,Double Jeopardy!,MAKE NO MYTHTAKE,$2000,He fell off Pegasus to his death,Bellerophon,he fell off pegasus to his death,bellerophon,2000,0.0,1.0
163776,5405,2008-02-22,Double Jeopardy!,"""B"" SHARP",$1200,This hero rode Pegasus,Bellerophon,this hero rode pegasus,bellerophon,1200,0.0,1.0
204878,6018,2010-11-10,Jeopardy!,MYTHOLOGY,$1000,He tamed the winged horse Pegasus with a bridle given to him by Athena,Bellerophon,he tamed the winged horse pegasus with a bridle given to him by athena,bellerophon,1000,0.0,1.0


Indeed we do. According to the sample of questions, there have been a total of **four** questions related to the mythological hero Bellerophon. It seems there is merit for contestants improving their knowledge of mythological figures.

In [23]:
jeopardy[jeopardy["Clean Answer"] == "aberdeen"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
140796,1237,1990-01-09,Jeopardy!,SCOTLAND,$500,"Scotland's ""Granite City""; its name means ""mouth of the Dee"" River, which is where it's located",Aberdeen,scotlands granite city its name means mouth of the dee river which is where its located,aberdeen,500,0.0,0.666667
151717,3273,1998-11-25,Double Jeopardy!,THERE'S SOMETHING ABOUT MARYLAND,$200,"This U.S. Army ""Proving Ground"" for weapons testing occupies over 70,000 acres in Harford County",Aberdeen,this us army proving ground for weapons testing occupies over 70000 acres in harford county,aberdeen,200,0.0,0.857143
210575,3773,2001-01-17,Jeopardy!,"""A"" IN GEOGRAPHY",$500,In the 1970s this Scottish fishing port became the center of the North Sea oil industry,Aberdeen,in the 1970s this scottish fishing port became the center of the north sea oil industry,aberdeen,500,0.0,1.000000
168430,4397,2003-10-21,Jeopardy!,"TAKE THE ""A"" TRAIN","$1,000",Scotrail's high speed Turbostar trains run on routes from Edinburgh to Glasgow & to this city,Aberdeen,scotrails high speed turbostar trains run on routes from edinburgh to glasgow to this city,aberdeen,0,0.0,0.666667
66790,4883,2005-11-30,Double Jeopardy!,EUROPEAN CITIES,$2000,"Known as the ""Granite City"", its name is Scots for ""At the Mouth of the Dee"", the river on which it lies",Aberdeen,known as the granite city its name is scots for at the mouth of the dee the river on which it lies,aberdeen,2000,0.0,1.000000
49613,5563,2008-11-12,Double Jeopardy!,"""A"" IN GEOGRAPHY",$2000,This Scottish seaport lies between the rivers Dee & Don,Aberdeen,this scottish seaport lies between the rivers dee don,aberdeen,2000,0.0,1.000000
169531,5776,2009-10-26,Double Jeopardy!,"""EEN""",$2000,"Scotland's third-largest city, it's known as the oil capital of Europe",Aberdeen,scotlands thirdlargest city its known as the oil capital of europe,aberdeen,2000,0.0,1.000000


`Aberdeen` has been the correct answer for **seven** questions in the dataset. Six of these seven occurrences have been related to the Scottish city. It is clear geographical knowledge is an important area of study, and this should not just be limited to US geography.

In [24]:
jeopardy[jeopardy["Clean Answer"] == "richard avedon"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
176983,2998,1997-09-17,Jeopardy!,"""A"" MEN",$500,This photographer known for his celebrity portraits learned his craft while in the Merchant Marine,Richard Avedon,this photographer known for his celebrity portraits learned his craft while in the merchant marine,richard avedon,500,0.0,1.0
210717,3593,2000-03-29,Double Jeopardy!,PHOTOGRAPHERS,$1000,"Truman Capote wrote the text for this Harper's Bazaar fashion photographer's 1959 collection ""Observances""",Richard Avedon,truman capote wrote the text for this harpers bazaar fashion photographers 1959 collection observances,richard avedon,1000,0.0,1.0
71255,4366,2003-09-08,Jeopardy!,PHOTOGRAPHERS,$600,"Dick Avery, Fred Astaire's character in ""Funny Face"", is based on this real-life photographer",Richard Avedon,dick avery fred astaires character in funny face is based on this reallife photographer,richard avedon,600,0.0,1.0
151383,4717,2005-02-22,Jeopardy!,"""A"" MEN",$600,This famed fashion photographer passed away in October 2004,(Richard) Avedon,this famed fashion photographer passed away in october 2004,richard avedon,600,0.0,1.0
6601,4985,2006-04-21,Jeopardy!,RICHARD,$800,He shot the famous photo of Nastassja Kinski & the serpent,Richard Avedon,he shot the famous photo of nastassja kinski the serpent,richard avedon,800,0.0,1.0
94950,5578,2008-12-03,Jeopardy!,PHOTOGRAPHY,$1000,"Truman Capote wrote the text for this fashion photographer's 1959 collection ""Observations""",Richard Avedon,truman capote wrote the text for this fashion photographers 1959 collection observations,richard avedon,1000,0.0,1.0


There have been six instances of questions related to the photographer `Richard Avedon` - there does indeed seem to be a trend of reoccurring questions, particularly in the context of historical people and figures (both from real life and fiction).

In [25]:
jeopardy[jeopardy["Clean Answer"] == "edward albee"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
92082,1197,1989-11-14,Double Jeopardy!,PLAYWRIGHTS,$400,"His first two produced plays were ""The Zoo Story"" and ""The Death of Bessie Smith""",Edward Albee,his first two produced plays were the zoo story and the death of bessie smith,edward albee,400,0.0,0.500000
54438,1255,1990-02-02,Double Jeopardy!,PLAYS,$600,"A very short-run play in 1963, written by D. Starkweather, was titled ""So Who's Afraid Of"" this playwright",Edward Albee,a very shortrun play in 1963 written by d starkweather was titled so whos afraid of this playwright,edward albee,600,0.0,0.666667
7932,1274,1990-03-01,Double Jeopardy!,AMERICAN PLAYS,$800,"This playwright dedicated ""A Delicate Balance"" to J. Steinbeck with ""affection and admiration""",Edward Albee,this playwright dedicated a delicate balance to j steinbeck with affection and admiration,edward albee,800,0.0,0.714286
27386,2339,1994-11-03,Final Jeopardy!,PLAYWRIGHTS,NaN,"He's won 3 Pulitzer Prizes for drama--in 1967, 1975 & 1994",Edward Albee,hes won 3 pulitzer prizes for dramain 1967 1975 1994,edward albee,0,0.0,0.666667
31634,2576,1995-11-13,Double Jeopardy!,PLAYS & PLAYWRIGHTS,$800,"The women in his play ""Three Tall Women"" are known by the letters ""A"", ""B"" & ""C"", not by names",Edward Albee,the women in his play three tall women are known by the letters a b c not by names,edward albee,800,0.0,1.000000
201075,2829,1996-12-12,Double Jeopardy!,PLAYWRIGHTS,$800,"Although his 1975 play ""Seascape"" had only a brief Broadway run, it won the Pulitzer Prize for Drama",Edward Albee,although his 1975 play seascape had only a brief broadway run it won the pulitzer prize for drama,edward albee,800,0.0,1.000000
176978,2998,1997-09-17,Jeopardy!,"""A"" MEN",$400,"In 1994 ""Three Tall Women"" earned this ""Seascape"" playwright his third Pulitzer Prize",Edward Albee,in 1994 three tall women earned this seascape playwright his third pulitzer prize,edward albee,400,0.0,1.000000
59703,3389,1999-05-06,Double Jeopardy!,NAME THE PLAYWRIGHT,$400,"""Who's Afraid of Virginia Woolf?""",Edward Albee,whos afraid of virginia woolf,edward albee,400,0.0,1.000000
77016,3490,1999-11-05,Jeopardy!,SCHOOL PLAYS,$400,"Some time passes before Jerry tells what happened at the zoo in this playwright's ""The Zoo Story""",Edward Albee,some time passes before jerry tells what happened at the zoo in this playwrights the zoo story,edward albee,400,0.0,1.000000
193209,4082,2002-05-07,Double Jeopardy!,PLAYBILL,$2000,"This playwright said that he's ""testing the limits of tolerance"" with his new play about 4 people & a goat",Edward Albee,this playwright said that hes testing the limits of tolerance with his new play about 4 people a goat,edward albee,2000,0.0,1.000000


Similarly, the American Playwright `Edward Albee` has been featured in 15 questions. There does indeed seem to be a pattern. It is certainly important for contestants to take note of which famous people and figures have appeared in past episodes as it is not out of the question that these questions will be recycled or slightly modified.

---
We could continue on looking into questions flagged to be similar, but to keep things concise, we can summarise the main key points:

* We estimate that around 1 in 20 questions have been asked before in some form or another.

* Looking into similar questions, it seems questions related to historical figures, people and geography are more frequently recycled. It would be worthwhile for contestants to take note of which famous people, countries and places appear as answers to questions as it is relatively likely that these will be answers to future questions in these categories.

## Low Value vs. High Value Questions

Here, we plan to determine if there is a relationship between terms used in questions and the question's monetary value. The `Chi-square` test can be used to answer this question.

To implement a Chi-square test:

* Questions can be classified as low-value (less than $800) or high-value (greater than $800)

* Looping through the set of all terms used (`terms_previously_used`): 

  - Find the number of low value questions each term occurs in.
  - Find the number of high value questions each term occurs in.
  - Find the percentage of questions the term occurs in.
  - Based on the percentage, find the expected count of questions the term occurs in.
  - Compute the Chi-square value from the observed and expected counts for high and low questions.

In [26]:
def determine_value(row):
  
  """Classifies each question as either a high value question or a low value question."""
  
  if row["Clean Value"] > 800:
    return 1
  
  else:
    return 0
  
jeopardy["High Value"] = jeopardy.apply(determine_value, axis=1)

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap,High Value
84523,1,1984-09-10,Jeopardy!,LAKES & RIVERS,$100,River mentioned most often in the Bible,the Jordan,river mentioned most often in the bible,the jordan,100,0.000000,0.0,0
84565,1,1984-09-10,Double Jeopardy!,THE BIBLE,$1000,"According to 1st Timothy, it is the ""root of all evil""",the love of money,according to 1st timothy it is the root of all evil,the love of money,1000,0.333333,0.0,1
84566,1,1984-09-10,Double Jeopardy!,'50'S TV,$1000,Name under which experimenter Don Herbert taught viewers all about science,Mr. Wizard,name under which experimenter don herbert taught viewers all about science,mr wizard,1000,0.000000,0.0,1
84567,1,1984-09-10,Double Jeopardy!,NATIONAL LANDMARKS,$1000,D.C. building shaken by November '83 bomb blast,the Capitol,dc building shaken by november 83 bomb blast,the capitol,1000,0.000000,0.0,1
84568,1,1984-09-10,Double Jeopardy!,NOTORIOUS,$1000,"After the deed, he leaped to the stage shouting ""Sic semper tyrannis""",John Wilkes Booth,after the deed he leaped to the stage shouting sic semper tyrannis,john wilkes booth,1000,0.000000,0.0,1


In [27]:
def count_usage(word):
  
  """Returns the number of high and low value questions that contain the given word."""

  low_count = 0
  high_count = 0
  
  for i, row in jeopardy.iterrows():
    split_question = row["Clean Question"].split(" ")
    if word in split_question:
      if row["High Value"] == 0:
        low_count += 1
        
      else:
        high_count += 1
        
  return high_count, low_count

A sample of ten single word terms are to be taken for the Chi-square test. This is due to how long it would take to perform the test on all single word terms in the question bank. 

In [30]:
from random import choice

terms_used_list = list(terms_previously_used)

comparison_terms = [choice(terms_used_list) for _ in range(10)]

observed_counts = []

for word in comparison_terms:
  observed_counts.append(count_usage(word))
  
observed_counts

[(0, 1),
 (0, 3),
 (0, 1),
 (57, 148),
 (0, 1),
 (0, 2),
 (0, 1),
 (2, 0),
 (0, 1),
 (1, 3)]

The observed counts for the ten selected terms are given above - (1,0) denotes a term that appears once in a high value question and zero times in low value questions.

---
Finally, we move to performing the Chi-square test using the observed values above. 

The expected values are calculated by finding the proportion of questions containing the given term and multiplying this by the total number of high value questions and total number of low value questions (to find the expected high value count and expected low value count, respectively).

In [31]:
from scipy.stats import chisquare

high_value_count = jeopardy["High Value"].sum()
low_value_count = len(jeopardy) - high_value_count

chi_square_values = []

for obs in observed_counts:
  total = sum(obs)
  total_prop = total / jeopardy.shape[0]
  high_value_expected = total_prop * high_value_count
  low_value_expected = total_prop * low_value_count
  
  observed = np.array([obs[0], obs[1]])
  expected = np.array([high_value_expected, low_value_expected])
  chi_square_values.append(chisquare(observed, expected))

chi_square_values

[Power_divergenceResult(statistic=np.float64(0.3235428703912728), pvalue=np.float64(0.5694862483821648)),
 Power_divergenceResult(statistic=np.float64(0.9706286111738185), pvalue=np.float64(0.3245234551241191)),
 Power_divergenceResult(statistic=np.float64(0.3235428703912728), pvalue=np.float64(0.5694862483821648)),
 Power_divergenceResult(statistic=np.float64(1.2528240516010078), pvalue=np.float64(0.26301378342263193)),
 Power_divergenceResult(statistic=np.float64(0.3235428703912728), pvalue=np.float64(0.5694862483821648)),
 Power_divergenceResult(statistic=np.float64(0.6470857407825455), pvalue=np.float64(0.4211565342143838)),
 Power_divergenceResult(statistic=np.float64(0.3235428703912728), pvalue=np.float64(0.5694862483821648)),
 Power_divergenceResult(statistic=np.float64(6.1815610326425166), pvalue=np.float64(0.012908834084008532)),
 Power_divergenceResult(statistic=np.float64(0.3235428703912728), pvalue=np.float64(0.5694862483821648)),
 Power_divergenceResult(statistic=np.float6

>Generally, the obtained p-values indicate there is a statistically insignificant difference between high value and low value questions at the 5% significance level (since almost all exceed 0.05) - it seems that there is no relationship between particular terms used in a question and whether the question is high or low value. 

In the case where a significant difference is suggested (a p-value of 0.0129), we must call this into question. This value originates from a term that has an observed tuple of (2,0) meaning the term appeared twice in high value questions and zero times in low value questions. This is simply not a large enough number of observed frequencies to reliably conclude anything.

Typically, for a reliable test, over 80% of the observed values in a contingency table should contain frequencies greater than 5. This would mean reliable conclusions can only really be drawn for terms that occur **frequently** across high and low value questions.

Nevertheless, it seems unlikely that there is any significant relationship between the occurrence of particular terms in a question and whether the question is high-valued or not. 

* It would instead be better to focus on preparing for the show by studying up on **general knowledge** - with a particular emphasis on US history, popular culture and wider geographical knowledge.